# HW4 Part 2 - Medical image fine-tuning

**Model:** ImageNet-pretrained ResNet-18. **Dataset:** PneumoniaMNIST (normal vs pneumonia pediatric chest X-rays). The convolutional backbone is frozen and the final classifier is fine-tuned. Run all cells in a Colab GPU runtime; the final cells report validation/test metrics, sensitivity, specificity, ROC AUC, and save plots.

In [ ]:
!pip -q install medmnist scikit-learn
import copy, json, random
from pathlib import Path
import matplotlib.pyplot as plt
import medmnist, numpy as np, torch, torch.nn as nn
from medmnist import INFO
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from torchvision import models, transforms

SEED, EPOCHS, BATCH_SIZE, IMAGE_SIZE = 42, 5, 64, 96
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])
normalize = transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
train_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.Grayscale(3), transforms.RandomHorizontalFlip(), transforms.RandomRotation(7), transforms.ToTensor(), normalize])
eval_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.Grayscale(3), transforms.ToTensor(), normalize])
train_ds = DataClass(split='train', transform=train_tf, download=True, size=28)
val_ds = DataClass(split='val', transform=eval_tf, download=True, size=28)
test_ds = DataClass(split='test', transform=eval_tf, download=True, size=28)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(len(train_ds), len(val_ds), len(test_ds), info['label'])

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters(): p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)

def epoch_pass(loader, training):
    model.train(training); total_loss = correct = count = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(device), y.squeeze(1).long().to(device)
            if training: optimizer.zero_grad(set_to_none=True)
            logits = model(x); loss = criterion(logits, y)
            if training: loss.backward(); optimizer.step()
            total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item(); count += len(y)
    return total_loss/count, correct/count

history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_acc, best_state = -1, None
for epoch in range(EPOCHS):
    tr_loss, tr_acc = epoch_pass(train_loader, True)
    va_loss, va_acc = epoch_pass(val_loader, False)
    for k,v in [('train_loss',tr_loss),('train_acc',tr_acc),('val_loss',va_loss),('val_acc',va_acc)]: history[k].append(v)
    if va_acc > best_acc: best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
    print(f'Epoch {epoch+1}: train_acc={tr_acc:.3f}, val_acc={va_acc:.3f}')
model.load_state_dict(best_state)

In [ ]:
model.eval(); ys, ps = [], []
with torch.no_grad():
    for x,y in test_loader:
        ps.append(torch.softmax(model(x.to(device)),1)[:,1].cpu().numpy()); ys.append(y.squeeze(1).numpy())
y = np.concatenate(ys); p = np.concatenate(ps); pred = (p>=0.5).astype(int)
cm = confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp = cm.ravel()
metrics = {'model':'ImageNet ResNet-18','dataset':'PneumoniaMNIST','best_validation_accuracy':best_acc,'test_accuracy':(tp+tn)/cm.sum(),'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp),'roc_auc':roc_auc_score(y,p),'confusion_matrix':cm.tolist()}
print(json.dumps(metrics, indent=2))
fpr,tpr,_ = roc_curve(y,p)
fig,ax = plt.subplots(1,3,figsize=(13,3.6))
ax[0].plot(history['train_loss'],label='train'); ax[0].plot(history['val_loss'],label='val'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].imshow(cm,cmap='Blues'); ax[1].set(title='Test confusion matrix',xlabel='Predicted',ylabel='True',xticks=[0,1],yticks=[0,1]); [ax[1].text(j,i,cm[i,j],ha='center',va='center') for i in range(2) for j in range(2)]
ax[2].plot(fpr,tpr,label=f"AUC={metrics['roc_auc']:.3f}"); ax[2].plot([0,1],[0,1],'--'); ax[2].set(title='Test ROC',xlabel='False-positive rate',ylabel='True-positive rate'); ax[2].legend()
plt.tight_layout(); plt.savefig('part2_results.png',dpi=180); plt.show()
Path('part2_metrics.json').write_text(json.dumps(metrics,indent=2)); torch.save(best_state,'resnet18_pneumoniamnist.pt')

## Brief interpretation

Report the best validation accuracy and held-out test metrics printed above. Sensitivity is the true-positive rate for pneumonia; specificity is the true-negative rate for normal images. Because this is a small educational dataset, these results are not evidence of clinical readiness.